## Step 1: Import Libraries

- **torch** = main PyTorch library  
- **nn** = neural network layers and modules  
- **optim** = optimizers like SGD or Adam  
- **datasets** and **transforms** = built-in datasets (MNIST) and preprocessing  
- **DataLoader** = batches the data for training  
- **Subset** = to take only 5000 samples


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

## Step 2: Load MNIST Dataset (5000 samples)

- MNIST images are 28x28 pixels.  
- We flatten them to a **784-length vector** for the MLP.  
- Use only **5000 samples** for faster training.  

Explanation:  
- **ToTensor()** converts image to tensor of shape (1,28,28)  
- **Normalize((0.5,), (0.5,))** scales pixels from [0,1] → [-1,1]  
- **Subset** selects first 5000 samples  
- **DataLoader** creates mini-batches of 64 samples for training


In [4]:
# Transforming images to tensors and normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download training data
full_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Taking only first 5000 samples
train_dataset = Subset(full_train_dataset, range(5000))

# DataLoader
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

100%|██████████| 9.91M/9.91M [00:00<00:00, 136MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 35.6MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 70.8MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.49MB/s]


## Step 3: Define the MLP Model

- **Input layer:** 28x28 = 784 features  
- **Hidden layers:** 128 → 64 neurons  
  - Each neuron acts like a "pattern detector"  
  - 128 neurons in first layer capture more features  
  - 64 neurons in second layer summarize features  
- **Output layer:** 10 neurons (digits 0-9)  
- **Activation function:** ReLU in hidden layers  

Why ReLU?  
- Introduces **non-linearity** so the network can learn complex patterns  
- Prevents vanishing gradient problems in deeper networks  

Output layer has **no activation** because `CrossEntropyLoss` applies **Softmax** internally.

In [5]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Initialize model
model = MLP()

## Step 4: Define Loss and Optimizer

- **Loss function:** CrossEntropyLoss  
  - Compares predicted outputs with true labels (0-9)  
- **Optimizer:** Adam  
  - Updates weights using gradients to minimize loss  
  - Adaptive learning rate, works well for beginner setups


In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Step 5: Training Loop

- **Forward pass:** compute predictions  
- **Compute loss:** compare prediction with true labels  
- **Backward pass:** compute gradients  
- **Update weights:** optimizer step  

We train for **5 epochs** to see the learning process.

In [7]:
epochs = 5

for epoch in range(epochs):
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [1/5], Loss: 0.4534
Epoch [2/5], Loss: 0.3811
Epoch [3/5], Loss: 0.6615
Epoch [4/5], Loss: 0.2112
Epoch [5/5], Loss: 0.1612


## Step 6: Testing

- Take a few test samples  
- Forward pass → get predictions  
- Compare predicted digits with actual


In [8]:
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=10, shuffle=True)

images, labels = next(iter(test_loader))
outputs = model(images)
_, predicted = torch.max(outputs, 1)

print("Predicted:", predicted)
print("Actual   :", labels)

Predicted: tensor([3, 7, 9, 4, 1, 7, 0, 8, 3, 0])
Actual   : tensor([9, 7, 9, 4, 1, 7, 0, 8, 3, 0])


### Summary

- **Layers:** 2 hidden layers  
- **Neurons:** 128 → 64  
- **Activation:** ReLU in hidden layers  
- **Dataset:** MNIST (5000 samples)  
- **Loss:** CrossEntropyLoss  
- **Optimizer:** Adam


## 📝 Assignment / Hands-On Practice

Now it’s your turn to experiment with the MLP model! Try the following:

1. **Increase the number of training samples**  
   - Currently we used 5000 samples. Can you use more from the MNIST dataset?

2. **Change the number of neurons in hidden layers**  
   - Try different configurations like 256 → 128 or 64 → 32  
   - Observe how it affects training and accuracy

3. **Add or remove hidden layers**  
   - What happens if you use only 1 hidden layer? Or 3 hidden layers?

4. **Change activation functions**  
   - Try `nn.Sigmoid()` or `nn.Tanh()` or `nn.ReLU()` 
   - Compare learning speed and loss

5. **Increase epochs**  
   - Currently trained for 5 epochs  
   - Try 10, 20, or more and observe how the loss decreases

6. **Optional:** Change optimizer or learning rate  
   - Try `optim.SGD` or different learning rates in Adam  
   - See how training behaves

💡 **Goal:** Understand how model capacity, activation functions, and dataset size affect learning. Record your observations and submit as a Word document along with the ipython notebook! You can above and beyond as well.

<b>Try to utilize your Google Colab Plus services it is free for students.</b>